# Practical Assignment 8: Recurrent Neural Network (RNN) from Scratch in NumPy

**Aim:** Build a Recurrent Neural Network by using the `numpy` library, implementing forward propagation, Backpropagation Through Time (BPTT), and training on sequential data.

**Reference:** BE IT Course: Lab Practice IV Lab Manual Assignment No. 7 (Pages 50-54).

**Mathematical Formulas:**
- Current Hidden State: $h_t = \text{sigmoid}(W_{xh} \cdot x_t + W_{hh} \cdot h_{t-1} + b_h)$
- Output State: $y_t = W_{ho} \cdot h_t + b_o$
- Backpropagation Through Time: Gradients are propagated backward through time steps.


In [1]:
import numpy as np

# Step 1: Activation Function and Derivative (from Lab Manual)
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

# Step 2: Define SimpleRNN Class from Scratch
class SimpleRNN:
    def __init__(self, input_size, hidden_size, output_size, learning_rate=0.05):
        self.hidden_size = hidden_size
        self.learning_rate = learning_rate
        
        # Learnable weight matrices and bias vectors
        self.W_xh = np.random.randn(input_size, hidden_size) * 0.1  # input to hidden
        self.W_hh = np.random.randn(hidden_size, hidden_size) * 0.1 # hidden to hidden
        self.W_ho = np.random.randn(hidden_size, output_size) * 0.1 # hidden to output
        self.b_h = np.zeros((1, hidden_size))
        self.b_o = np.zeros((1, output_size))
        
    def forward(self, inputs):
        self.hidden_states = {}
        self.outputs = {}
        self.h_prev = np.zeros((1, self.hidden_size))  # initial state
        
        for t, x in enumerate(inputs):
            x = x.reshape(1, -1)
            # h_t = sigmoid(x * W_xh + h_prev * W_hh + b_h)
            self.h_prev = sigmoid(np.dot(x, self.W_xh) + np.dot(self.h_prev, self.W_hh) + self.b_h)
            self.hidden_states[t] = self.h_prev
            # y_t = h_t * W_ho + b_o
            output = np.dot(self.h_prev, self.W_ho) + self.b_o
            self.outputs[t] = output
            
        return self.outputs[len(inputs) - 1]  # Return final output
        
    def backward(self, inputs, targets):
        dW_xh, db_h = np.zeros_like(self.W_xh), np.zeros_like(self.b_h)
        dW_hh, dW_ho = np.zeros_like(self.W_hh), np.zeros_like(self.W_ho)
        db_o = np.zeros_like(self.b_o)
        dh_next = np.zeros((1, self.hidden_size))
        
        for t in reversed(range(len(inputs))):
            x = inputs[t].reshape(1, -1)
            h_t = self.hidden_states[t]
            h_prev_t = self.hidden_states[t - 1] if t > 0 else np.zeros((1, self.hidden_size))
            
            # Output error gradient
            d_output = 2 * (self.outputs[t] - targets[t].reshape(1, -1))
            dW_ho += np.dot(h_t.T, d_output)
            db_o += d_output
            
            # Hidden state gradient through time (BPTT)
            dh = np.dot(d_output, self.W_ho.T) + dh_next
            dh_raw = dh * sigmoid_derivative(h_t)
            
            dW_xh += np.dot(x.T, dh_raw)
            db_h += dh_raw
            dW_hh += np.dot(h_prev_t.T, dh_raw)
            dh_next = np.dot(dh_raw, self.W_hh.T)
            
        # Update weights and biases via Gradient Descent
        self.W_xh -= self.learning_rate * dW_xh
        self.b_h -= self.learning_rate * db_h
        self.W_hh -= self.learning_rate * dW_hh
        self.W_ho -= self.learning_rate * dW_ho
        self.b_o -= self.learning_rate * db_o

# Step 3: Train the RNN on Sequential Data
input_seq = np.array([[0.1], [0.2], [0.3], [0.4], [0.5]])
target_seq = np.array([[0.2], [0.4], [0.6], [0.8], [1.0]])

rnn = SimpleRNN(input_size=1, hidden_size=5, output_size=1, learning_rate=0.05)
print('Training NumPy SimpleRNN...')
for epoch in range(1, 301):
    pred = rnn.forward(input_seq)
    loss = np.mean((pred - target_seq[-1]) ** 2)
    rnn.backward(input_seq, target_seq)
    if epoch % 100 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d} | MSE Loss: {loss:.6f}')

# Step 4: Test with new sequence
test_seq = np.array([[0.6], [0.7], [0.8], [0.9], [1.0]])
test_pred = rnn.forward(test_seq)
print(f'\nTest Sequence Output Prediction: {test_pred[0][0]:.4f}')


Training NumPy SimpleRNN...
Epoch   1 | MSE Loss: 1.360992
Epoch 100 | MSE Loss: 0.139948
Epoch 200 | MSE Loss: 0.088631
Epoch 300 | MSE Loss: 0.050129

Test Sequence Output Prediction: 0.8331
